## Mission and Dataset

This notebook trains regression models to predict the Air Quality Index (AQI) using pollutant measurements from multiple Indian cities.

For the full mission description, dataset details, and source, see the project `README.md`. 

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib, os

os.makedirs("outputs", exist_ok=True)
np.random.seed(42)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
pip install kagglehub[pandas-datasets]

In [ ]:
# Download the dataset locally (cached by kagglehub) and read with pandas
cache_dir = kagglehub.dataset_download("rohanrao/air-quality-data-in-india")
csv_path = os.path.join(cache_dir, "city_day.csv")

df = pd.read_csv(
    csv_path,
    encoding="latin1",
    encoding_errors="ignore",
    sep=",",
    on_bad_lines="skip",
)

df["Year"] = pd.to_datetime(df["Date"], errors="coerce").dt.year
df = df.drop(columns=["Date", "AQI_Bucket"], errors="ignore")

print(f"Shape: {df.shape}\n")
print(df.head())
print("\nMissing values:\n", df.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

os.makedirs("outputs", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df["AQI"].dropna().hist(bins=50, ax=axes[0], color="#f97316")
axes[0].set(title="AQI Distribution", xlabel="AQI", ylabel="Count")

city_aqi = df.groupby("City")["AQI"].median().sort_values()
city_aqi.plot.barh(ax=axes[1], color="steelblue")
axes[1].set(title="Median AQI by City", xlabel="Median AQI")
plt.tight_layout()
plt.show()
plt.savefig("outputs/fig1_distribution.png", dpi=120)
plt.close()

# Fig 2: Correlation heatmap
num_cols = [c for c in ["PM2.5","PM10","NO","NO2","NOx","NH3","SO2","CO","O3","Benzene","Toluene","Xylene","AQI"] if c in df.columns]
plt.figure(figsize=(11, 9))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.4)
plt.title("Pollutant Correlation Matrix")
plt.tight_layout()
plt.show()
plt.savefig("outputs/fig2_correlation.png", dpi=120)
plt.close()

# Fig 3: Top pollutants vs AQI
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, feat, color in zip(axes.flatten(), ["PM2.5","PM10","NO2","SO2"], ["#f97316","#3b82f6","#10b981","#a855f7"]):
    tmp = df[[feat, "AQI"]].dropna()
    ax.scatter(tmp[feat], tmp["AQI"], alpha=0.3, s=10, color=color)
    m, b = np.polyfit(tmp[feat], tmp["AQI"], 1)
    ax.plot(np.sort(tmp[feat]), m * np.sort(tmp[feat]) + b, "w-", lw=2)
    ax.set(title=f"{feat} vs AQI  (R²={np.corrcoef(tmp[feat], tmp['AQI'])[0,1]**2:.3f})",
           xlabel=feat, ylabel="AQI")
plt.tight_layout()
plt.show()
plt.savefig("outputs/fig3_scatter_pollutants.png", dpi=120)
plt.close()

print("\nEDA figures saved.")

## Exploratory Visualizations (Summary)

- AQI is right‑skewed with some extreme high‑pollution days.
- Cities have different median AQI values, which motivates using a city‑level feature.
- AQI is strongly correlated with PM10, PM2.5 and CO, and moderately with NO2 and SO2.
- Scatter plots confirm that higher pollutant levels generally correspond to higher AQI.

More detailed discussion is in `README.md`. 

In [ ]:
print("\nCorrelation with AQI:\n", df[num_cols].corr()["AQI"].sort_values(ascending=False))

# Drop: NOx (multicollinear with NO+NO2), Toluene/Xylene (low correlation), Year (weak predictor)
DROP = [c for c in ["NOx","Toluene","Xylene","Year"] if c in df.columns]
df2 = df.drop(columns=DROP)

# Encode City as ordinal (ranked by city mean AQI)
df2["City_encoded"] = df2["City"].map(df2.groupby("City")["AQI"].mean().rank())
df2 = df2.drop(columns=["City"])

# Impute with median, then drop remaining NaNs
df2 = df2.apply(lambda col: col.fillna(col.median()))
df2 = df2.dropna()

FEATURES = [c for c in df2.columns if c != "AQI"]
print(f"\nFeatures used ({len(FEATURES)}): {FEATURES}")
print(f"Clean dataset: {df2.shape}")

## Feature Engineering (Summary)

- Dropped NOx, Toluene, Xylene and Year based on correlation analysis and weak predictive value.
- Encoded City as `City_encoded` (ranked by mean AQI) to capture city‑level pollution differences.
- Imputed missing numeric values with the median and standardized features using `StandardScaler`.

Full reasoning is documented in `README.md`. 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    df2[FEATURES].values, df2["AQI"].values, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

In [ ]:
# Use the saved best model to predict AQI for one row of the test set
artifact = joblib.load("outputs/best_aqi_model.pkl")
model = artifact["model"]
scaler = artifact["scaler"]
features = artifact["features"]

# Take one row from the held‑out test data and predict
one_row_scaled = X_test_sc[[0]]  # shape (1, n_features)
predicted_aqi = model.predict(one_row_scaled)[0]
actual_aqi = y_test[0]

print(f"Predicted AQI for first test sample: {predicted_aqi:.2f}")
print(f"Actual AQI for first test sample:    {actual_aqi:.2f}")

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np



train_loss, test_loss = [], []
sgd = SGDRegressor(loss="squared_error", learning_rate="constant",
                   eta0=0.0001, max_iter=1, warm_start=True, random_state=42, tol=None)

for _ in range(100):
    sgd.fit(X_train_sc, y_train)
    train_loss.append(mean_squared_error(y_train, sgd.predict(X_train_sc)))
    test_loss.append(mean_squared_error(y_test,  sgd.predict(X_test_sc)))

# Loss curve
plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Train MSE"); plt.plot(test_loss, "--", label="Test MSE")
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.title("Gradient Descent — Loss Curve")
plt.legend(); plt.tight_layout(); plt.savefig("outputs/fig4_loss_curve.png", dpi=120); plt.close()

# Before/After scatter
sgd1 = SGDRegressor(loss="squared_error", learning_rate="constant",
                    eta0=0.0001, max_iter=1, random_state=42, tol=None)
sgd1.fit(X_train_sc, y_train)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, preds, title in zip(axes,
    [sgd1.predict(X_test_sc), sgd.predict(X_test_sc)],
    ["BEFORE (Epoch 1)", "AFTER (Epoch 100)"]):
    ax.scatter(y_test, preds, alpha=0.35, s=12)
    lim = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lim, lim, "r--", lw=2, label="Perfect fit")
    ax.set(title=f"{title} — R²={r2_score(y_test, preds):.4f}",
           xlabel="Actual AQI", ylabel="Predicted AQI")
    ax.legend()
plt.tight_layout(); plt.savefig("outputs/fig5_before_after.png", dpi=120); plt.close()

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=8, min_samples_leaf=10, random_state=42)
dt.fit(X_train_sc, y_train)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import matplotlib.pyplot as plt

rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=5,
                            n_jobs=-1, random_state=42)
rf.fit(X_train_sc, y_train)

# Feature importance
pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh(figsize=(9,5), color="steelblue")
plt.title("Feature Importances — Random Forest")
plt.tight_layout(); plt.savefig("outputs/fig6_feature_importance.png", dpi=120); plt.close()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pandas as pd
import matplotlib.pyplot as plt
import joblib

def evaluate(model, name):
    p = model.predict(X_test_sc)
    return {"Model": name,
            "Train R²": r2_score(y_train, model.predict(X_train_sc)),
            "Test R²":  r2_score(y_test, p),
            "MAE":      mean_absolute_error(y_test, p),
            "RMSE":     mean_squared_error(y_test, p) ** 0.5}

results = pd.DataFrame([evaluate(sgd, "Linear Regression (GD)"),
                         evaluate(dt,  "Decision Tree"),
                         evaluate(rf,  "Random Forest")])
print("\n", results.to_string(index=False))

# Bar chart comparison
results.set_index("Model")[["Test R²","MAE","RMSE"]].plot.bar(figsize=(10, 5), rot=15)
plt.title("Model Comparison"); plt.tight_layout()
plt.savefig("outputs/fig7_model_comparison.png", dpi=120); plt.close()

# Save best model
best_name = results.loc[results["Test R²"].idxmax(), "Model"]
best_model = {"Linear Regression (GD)": sgd, "Decision Tree": dt, "Random Forest": rf}[best_name]
joblib.dump({"model": best_model, "scaler": scaler, "features": FEATURES}, "outputs/best_aqi_model.pkl")
print(f"\nBest model: {best_name} → saved to outputs/best_aqi_model.pkl")